# C2 — ACF Lag-1 per Roll Day

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for wk, wm in WINDOWS_META.items():
    rk = wm['result_key']
    ts = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'timeseries.parquet')
    ts.index = pd.to_datetime(ts.index)
    ts['date'] = ts.index.date.astype(str)
    rhos = []
    for d in wm['days']:
        sub = ts[ts['date'] == d]['dev'].dropna().values
        if len(sub) < 3:
            rhos.append(np.nan); continue
        rho = np.corrcoef(sub[1:], sub[:-1])[0,1]
        rhos.append(rho)
    ax.plot(range(len(wm['days'])), rhos, marker='o', label=wk, color=WIN_COLORS[wk])
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(range(7))
ax.set_xticklabels(['D1','D2','D3','D4','D5','D6','D7'])
ax.set_xlabel('Roll Day')
ax.set_ylabel('ACF Lag-1 (rho)')
ax.set_title('ACF Lag-1 per Roll Day — All Windows (negative = mean-reverting)')
ax.legend()
save_fig(fig, 'C2_acf_lag1_per_rollday.png')
